## Phase 2: Vision Extraction (Week2)

In [4]:
!pip install anthropic

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [anthropic]/7 [anthropic]parser]


In [5]:
import anthropic
import base64
import json
import boto3

In [6]:
ANTHROPIC_API_KEY = "sk-"

# load image
image_path = "/home/ec2-user/SageMaker/test1.png"
with open(image_path, "rb") as f:
    image_data = base64.standard_b64encode(f.read()).decode("utf-8")

# Vision extraction prompt
extraction_prompt = """Analyze this Human Design BodyGraph chart image. Extract the following information and return ONLY a JSON object with no other text:

{
  "type": "The Human Design Type (Generator, Manifesting Generator, Projector, Manifestor, or Reflector)",
  "authority": "The Inner Authority (e.g., Emotional/Solar Plexus, Sacral, Splenic, etc.)",
  "profile": "The Profile numbers (e.g., 4/6, 1/3, etc.)",
  "strategy": "The Strategy (e.g., To Respond, To Wait for the Invitation, To Inform, To Wait a Lunar Cycle)",
  "definition": "The Definition type (Single, Split, Triple Split, Quadruple Split, No Definition)",
  "defined_centers": ["List of all colored/defined centers"],
  "undefined_centers": ["List of all white/undefined centers"],
  "active_channels": ["List of active channels as gate-gate pairs"]
}

IMPORTANT RULES:
- Colored/shaded centers (any color: red, orange, yellow, brown, purple) are DEFINED
- White/empty centers are UNDEFINED
- The 9 centers are: Head, Ajna, Throat, G/Self, Heart/Will/Ego, Sacral, Solar Plexus/Emotional, Spleen, Root
- Channels are the lines connecting two centers - if a channel is fully colored, it is active
- Look at the Design (left) and Personality (right) columns for gate numbers
- Return ONLY valid JSON, no explanation"""

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1500,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": "image/png",
                        "data": image_data,
                    },
                },
                {
                    "type": "text",
                    "text": extraction_prompt
                }
            ],
        }
    ],
)

# parse response
result_text = response.content[0].text
print(result_text)

```json
{
  "type": "Generator",
  "authority": "Emotional/Solar Plexus",
  "profile": "4/6",
  "strategy": "To Respond",
  "definition": "Split",
  "defined_centers": ["Throat", "G/Self", "Sacral", "Solar Plexus/Emotional", "Root"],
  "undefined_centers": ["Head", "Ajna", "Heart/Will/Ego", "Spleen"],
  "active_channels": ["34-57", "14-2", "27-50", "41-30"]
}
```


In [8]:
with open("/home/ec2-user/SageMaker/test1.png", "rb") as f:
    img_b64 = base64.standard_b64encode(f.read()).decode("utf-8")

lambda_client = boto3.client("lambda", region_name="us-east-1")

response = lambda_client.invoke(
    FunctionName="human-design-vision",
    InvocationType="RequestResponse",
    Payload=json.dumps({
        "image": img_b64,
        "media_type": "image/png"
    })
)

result = json.loads(response["Payload"].read().decode("utf-8"))
result

{'statusCode': 200,
 'headers': {'Content-Type': 'application/json'},
 'body': '{"success": true, "chart_data": {"type": "Generator", "authority": "Emotional/Solar Plexus", "profile": "1/3", "strategy": "To Respond", "definition": "Single", "defined_centers": ["Throat", "G/Self", "Sacral", "Solar Plexus/Emotional", "Root"], "undefined_centers": ["Head", "Ajna", "Heart/Will/Ego", "Spleen"], "active_channels": ["14-2", "34-57", "27-50", "41-30"]}}'}